# StormEngine V6 — Encoder–Decoder Reconstruction Diagnostic

This experiment bypasses the ConvGRU Processor and asks a narrower question: can the Encoder and Decoder reconstruct the **current** ERA5 grid from sparse values at the 390 project coordinates? It uses 2010–2015 for training and 2016 for validation. It does not read the 2017 held-out test set.

Targets are `msl`, `u10`, `v10`, and `t2m`. Precipitation is excluded because V6 does not provide `tp` at the sparse input points; including it would mix a missing-input problem into the spatial reconstruction diagnosis.

In [ ]:
from pathlib import Path
import json, subprocess, sys

here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO

def run_live(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

CONFIG = REPO / 'configs' / 'era5_2010_2017_windows.local.yaml'
assert CONFIG.exists(), 'Run the configuration cell in StormEngine_V6_EndToEnd.ipynb first.'
print('Repository:', REPO)
print('Config:', CONFIG)

## 1. Verify CUDA

In [ ]:
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Smoke run

Two training batches and one validation batch test data alignment, forward/backward propagation, and artifact writing. These numbers are not scientific results.

In [ ]:
run_live([
    sys.executable, '-u', str(REPO / 'scripts' / 'train_reconstruction.py'),
    '--config', str(CONFIG), '--device', DEVICE, '--epochs', '1',
    '--max-train-batches', '2', '--max-eval-batches', '1',
    '--output-dir', 'artifacts/v6_reconstruction_smoke'
])

## 3. Medium pilot

This short run checks whether training and validation losses decrease before committing to all epochs. It still uses only a subset of batches.

In [ ]:
run_live([
    sys.executable, '-u', str(REPO / 'scripts' / 'train_reconstruction.py'),
    '--config', str(CONFIG), '--device', DEVICE, '--epochs', '3',
    '--max-train-batches', '200', '--max-eval-batches', '50',
    '--output-dir', 'artifacts/v6_reconstruction_pilot'
])

## 4. Full reconstruction diagnostic

Run this only if the pilot behaves normally. The configured maximum is 80 epochs, with early stopping after 12 validation epochs without improvement. The best model is selected using 2016 validation loss only.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    run_live([
        sys.executable, '-u', str(REPO / 'scripts' / 'train_reconstruction.py'),
        '--config', str(CONFIG), '--device', DEVICE,
        '--output-dir', 'artifacts/v6_reconstruction'
    ])
else:
    print('Full run locked. Set RUN_FULL=True after inspecting the pilot.')

## 5. Inspect curves and validation metrics

Choose `v6_reconstruction_pilot` first. After the full run, change it to `v6_reconstruction`. These are 2016 validation results, intended for component diagnosis and model decisions.

In [ ]:
import matplotlib.pyplot as plt

RESULT_NAME = 'v6_reconstruction_pilot'
result_dir = REPO / 'artifacts' / RESULT_NAME
history = json.loads((result_dir / 'history.json').read_text(encoding='utf-8'))
metrics = json.loads((result_dir / 'validation_metrics.json').read_text(encoding='utf-8'))
epochs = [row['epoch'] for row in history]
plt.figure(figsize=(8, 4))
plt.plot(epochs, [row['train_loss'] for row in history], marker='o', label='train')
plt.plot(epochs, [row['validation_loss'] for row in history], marker='o', label='2016 validation')
plt.xlabel('Epoch'); plt.ylabel('Sea-weighted normalized MSE'); plt.grid(alpha=.3); plt.legend(); plt.show()
print(json.dumps(metrics, indent=2))

## Interpretation

Compare this model's 2016 simultaneous-reconstruction errors with the V6 forecast's short-lead validation errors. If reconstruction is already weak over the sea, prioritize station representation, coordinate encoding, Gaussian spreading, and Decoder design. If reconstruction is strong but +1 h/+2 h forecasts degrade sharply, the temporal Processor and autoregressive rollout become the primary suspects. This experiment is diagnostic; it does not replace the final forecasting model.